In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import os

In [2]:
df=pd.read_csv("/kaggle/input/datasets/aman99842/wordac/word_com.csv")

In [3]:
df.head(4)

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen


In [4]:
quotes=df['quote']

In [5]:
quotes.head()

0    “The world as we have created it is a process ...
1    “It is our choices, Harry, that show what we t...
2    “There are only two ways to live your life. On...
3    “The person, be it gentleman or lady, who has ...
4    “Imperfection is beauty, madness is genius and...
Name: quote, dtype: object

In [6]:
quotes=quotes.str.lower()

In [7]:
# remove puncutation
import string

def remove_punc(txt):
    return txt.translate(str.maketrans('','',string.punctuation))

quotes=quotes.apply(remove_punc)


In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer
vocab_size=8980
tokenizer=Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)

2026-03-28 18:30:44.037065: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774722644.218781      25 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774722644.275245      25 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774722644.684005      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774722644.684053      25 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774722644.684055      25 computation_placer.cc:177] computation placer alr

In [9]:
word_index=tokenizer.word_index
print(len(word_index)) 
# print(word_index,slice, 12)

8978


In [10]:
input_sequences = []

for line in quotes:
    token_list = tokenizer.texts_to_sequences([line])[0]
    
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

In [11]:
input_sequences[0]

[713, 62]

In [12]:
# X=[]
# y=[]

# for seq in sequence:
#     for i in range(1,len(seq)):
#         input_seq=seq[:i]
#         output_seq=seq[i]
#         X.append(input_seq)
#         y.append(output_seq)


In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

maxlen = max(len(x) for x in input_sequences)

input_sequences = pad_sequences(input_sequences, maxlen=maxlen, padding='pre')

In [14]:
X = input_sequences[:, :-1]
y = input_sequences[:, -1]

In [15]:
from tensorflow.keras.utils import to_categorical

y = to_categorical(y, num_classes=vocab_size)

In [16]:
y

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [17]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

lstm_model = Sequential([
    Embedding(vocab_size, 100, input_length=maxlen),
    LSTM(150),
    Dense(vocab_size, activation='softmax')
])

lstm_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(
I0000 00:00:1774722664.129826      25 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


In [18]:
lstm_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
lstm_model.fit(X, y, epochs=40, batch_size=64)

Epoch 1/40


I0000 00:00:1774722677.256370      70 cuda_dnn.cc:529] Loaded cuDNN version 91002


1333/1333 ━━━━━━━━━━━━━━━━━━━━ 75s 53ms/step - accuracy: 0.0397 - loss: 6.8829
Epoch 2/40
1333/1333 ━━━━━━━━━━━━━━━━━━━━ 71s 54ms/step - accuracy: 0.0838 - loss: 6.0607
Epoch 3/40
1333/1333 ━━━━━━━━━━━━━━━━━━━━ 71s 53ms/step - accuracy: 0.1113 - loss: 5.6288
Epoch 4/40
1333/1333 ━━━━━━━━━━━━━━━━━━━━ 72s 54ms/step - accuracy: 0.1299 - loss: 5.3172
Epoch 5/40
1333/1333 ━━━━━━━━━━━━━━━━━━━━ 72s 54ms/step - accuracy: 0.1476 - loss: 5.0136
Epoch 6/40
1333/1333 ━━━━━━━━━━━━━━━━━━━━ 72s 54ms/step - accuracy: 0.1638 - loss: 4.7193
Epoch 7/40
1333/1333 ━━━━━━━━━━━━━━━━━━━━ 72s 54ms/step - accuracy: 0.1837 - loss: 4.4570
Epoch 8/40
1333/1333 ━━━━━━━━━━━━━━━━━━━━ 72s 54ms/step - accuracy: 0.2059 - loss: 4.2161
Epoch 9/40
1333/1333 ━━━━━━━━━━━━━━━━━━━━ 72s 54ms/step - accuracy: 0.2330 - loss: 3.9662
Epoch 10/40
1333/1333 ━━━━━━━━━━━━━━━━━━━━ 71s 54ms/step - accuracy: 0.2633 - loss: 3.7403
Epoch 11/40
1333/1333 ━━━━━━━━━━━━━━━━━━━━ 72s 54ms/step - accuracy: 0.2920 - loss: 3.5365
Epoch 12/40
1333/13

In [20]:
lstm_model.save("/kaggle/working/lstm_model.keras")

In [21]:
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_next_word(text):
    seq = tokenizer.texts_to_sequences([text])[0]
    seq = pad_sequences([seq], maxlen=maxlen-1, padding='pre')
    
    pred = lstm_model.predict(seq)
    word_index = np.argmax(pred)
    
    for word, index in tokenizer.word_index.items():
        if index == word_index:
            return word

In [22]:
print(predict_next_word("where are you"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step
i
